# 01. 이메일 내용으로부터 구조화된 정보 추출하기

In [1]:
email_conversation = '''From: 테디 (teddy@teddynote.com)
To: 이은채 대리님 (eunchae@teddyinternaltional.me)
Subject: RAG 솔루션 시연 관련 미팅 제안

안녕하세요, 이은채 대리님,

저는 테디노트의 테디입니다. 최근 귀사에서 AI를 활용한 혁신적인 솔루션을 모색 중이라는 소식을 들었습니다. 테디노트는 AI 및 RAG 솔루션 분야에서 다양한 경험과 노하우를 가진 기업으로, 귀사의 요구에 맞는 최적의 솔루션을 제공할 수 있다고 자부합니다.

저희 테디노트의 RAG 솔루션은 귀사의 데이터 활용을 극대화하고, 실시간으로 정확한 정보 제공을 통해 비즈니스 의사결정을 지원하는 데 탁월한 성능을 보입니다. 이 솔루션은 특히 다양한 산업에서의 성공적인 적용 사례를 통해 그 효과를 입증하였습니다.

귀사와의 협력 가능성을 논의하고, 저희 RAG 솔루션의 구체적인 기능과 적용 방안을 시연하기 위해 미팅을 제안드립니다. 다음 주 목요일(7월 18일) 오전 10시에 귀사 사무실에서 만나 뵐 수 있을까요?

미팅 시간을 조율하기 어려우시다면, 편하신 다른 일정을 알려주시면 감사하겠습니다. 이은채 대리님과의 소중한 만남을 통해 상호 발전적인 논의가 이루어지길 기대합니다.

감사합니다.

테디
테디노트 AI 솔루션팀'''

In [2]:
from pydantic import BaseModel, Field

class EmailSummary(BaseModel):
    person: str = Field(description='메일을 보낸 사람')
    company: str = Field(description='메일을 보낸 사람의 회사 정보')
    email: str = Field(description='메일을 보낸 사람의 이메일 주소')
    subject: str = Field(description='메일 제목')
    summary: str = Field(description='메일 본문을 요약한 텍스트')
    date: str = Field(description='메일 본문에 언급된 미팅 날짜와 시간')

In [3]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(temperature=0, model_name='gpt-4o')

In [4]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [5]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    '''
You are a helpful assistant. Please answer the following questions in KOREAN.

#QUESTION:
다음의 이메일 내용 중에서 주요 내용을 추출해 주세요.

#EMAIL CONVERSATION:
{email_conversation}

#FORMAT:
{format}
'''
)

prompt = prompt.partial(format=output_parser.get_format_instructions())

In [6]:
chain = prompt | llm | output_parser

answer = chain.invoke({'email_conversation': email_conversation})
print(answer.summary)

테디노트의 테디가 이은채 대리님에게 AI 및 RAG 솔루션 시연을 위한 미팅을 제안하며, 솔루션의 기능과 적용 방안을 설명하고 협력 가능성을 논의하고자 한다.


# 02. SerpAPI를 정보 검색에 활용하기

In [7]:
load_dotenv()

from langchain_community.utilities import SerpAPIWrapper

params = {'engine': 'google', 'gl': 'kr', 'hl': 'ko', 'num': '3'}

search = SerpAPIWrapper(params=params)

In [8]:
search.run('테디노트')

'[\'데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 \\u200d♂️ ...more 데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 ...\', \'블로그와 유튜브 "테디노트"를 운영하고 있으며, "파이썬 딥러닝 텐서플로"를 집필하였습니다. 데이터분석과 AI를 사랑하고 지식공유에 활발히 참여하고 있습니다.\', \'경력 · Ambassador · Content Creator · CEO / Founder · Co-founder/CTO · Software Developer. 삼성전자. 2014년 1월 - 2016년 10월 2년 10개월. Suwon.\', \'데이터와 인공지능을 좋아하는 개발자 노트.\', \'Teddy Lee (이경록) ... 기업용 AI 솔루션 · AI 교육 · 에이전트 빌더 플랫폼을 만듭니다. LangChain Global Ambassador · 테디노트 운영 · RAG / Agent 시스템 빌더.\', "RAG 기본기부터 심화까지 제대로 끝내는 45시간 완성 로드맵, 테디노트의 \'진짜\' RAG 활용법!", \'8단계 RAG 파이프라인으로 LLM 성능을 끌어올리는 RAG 기본부터 실제 챗봇 제작까지 | PDF도 App에서 필기하며 독서하세요! 이경록 저자 ...\', \'본 저작물은 2025년 테디노트에 의해 작성되었습니다. 모든 권리는 저작권자에게 있으며, 본 저작물은 Creative Commons Attribution-NonCommercial- ...\', \'테디노트 X 패스트캠퍼스 "RAG 비법노트" · 환경 설정 (Mac) · 환경 설정 (Windows). LocalModels. GGUF · HuggingFace gguf 파일을 Ollama 로딩.\']'

In [9]:
search.run('테디노트 site:github.com')

'[\'Teddy Lee (이경록) ... 기업용 AI 솔루션 · AI 교육 · 에이전트 빌더 플랫폼을 만듭니다. LangChain Global Ambassador · 테디노트 운영 · RAG / Agent 시스템 빌더.\', \'랭체인 한국어 튜토리얼에 사용되는 다양한 유틸 파이썬 패키지. LangChain 을 사용하면서 불편한 기능이나, 추가적인 기능을 제공합니다. 다운로드 통계. 설치.\', \'Teddy Lee (이경록) ... 기업용 AI 솔루션 · AI 교육 · 에이전트 빌더 플랫폼을 만듭니다. LangChain Global Ambassador · 테디노트 운영 · RAG / Agent 시스템 빌더.\', \'Forked from teddylee777/machine-learning. 머신러닝 입문자 혹은 스터디를 준비하시는 분들에게 도움이 되고자 만든 repository입니다.\', \'LangChain 밋업 2024 Q1 발표자료 · RAG - 우리가 절대 쉽게 원하는 결과물을 얻을 수 없는 이유 - 테디노트 · 프름프트 흐름과 LLM 모델 평가 - 이재석님 · 인공지능을 ...\', \'테디노트 YouTube 로 RAG 배우기! RAG 비법노트. 소개. 이 프로젝트는 LangGraph를 사용하여 AI 에이전트를 구축하고 실행하는 방법 ...\', \'LangChain 을 활용한 어플리케이션 제작. @author: 테디노트. 랭체인 온라인 교재 · 랭체인 한국어 튜토리얼 · YouTube 테디노트. 위키독스 전자책(무료). 위키독스에 ...\', \'학습 자료. 테디노트 유튜브 채널 - AI/ML 관련 한국어 강의 및 튜토리얼; RAG 고급 온라인 강의 - 체계적인 RAG 시스템 구축 강의. 라이센스. 본 프로젝트의 ...\', "Teddy Lee teddylee777.. Working from home. YouTube \'테디노트\' Creator. LangChain Ambassador. Specializing in LLM applications

In [10]:
answer.email

'teddy@teddynote.com'

In [11]:
query = f'{answer.person} {answer.company} {answer.email}'
query

'테디 테디노트 teddy@teddynote.com'

In [12]:
search.run(query)

'[\'데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 \\u200d♂️ ...more 데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 ...\', \'테디노트 X 패스트캠퍼스 "RAG 비법노트" · 환경 설정 (Mac) · 환경 설정 (Windows). LocalModels. GGUF · HuggingFace gguf 파일을 Ollama 로딩 · TeddyNote.\', \'이 글에서는 LangChain 의 Agent 프레임워크를 활용하여 복잡한 검색과 데이터 처리 작업을 수행하는 방법을 소개합니다. LangSmith 를 사용하여 Agent의 추론 단계를 추적 ...\', \'Teddy Lee (이경록) ... 기업용 AI 솔루션 · AI 교육 · 에이전트 빌더 플랫폼을 만듭니다. LangChain Global Ambassador · 테디노트 운영 · RAG / Agent 시스템 빌더.\', \'테디노트 ; 한줄 소개: 블로그와 유튜브 "테디노트"를 운영하고 있으며, "파이썬 딥러닝 텐서플로"를 집필하였습니다. 데이터분석과 AI를 사랑하고 지식공유에 활발히 참여 ...\', \'경력 ; Ambassador. LangChain. 2025년 2월 – 현재 1년 6개월. 대한민국 서울 ; Content Creator. 테디노트 TeddyNote. 2020년 5월 – 현재 6년 3개월 ; CEO / Founder.\', \'환경 설정 (Windows) · git 설치 · PowerShell Policy 적용 · pyenv 설치 · python 설치 · Poetry 설치 · 실습코드 다운로드 · Visual Studio Code 설치.\', \'데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 \\u200d♂️ "테디노트의 RAG 비법노트" 랭체인 강의: ...\', \'데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에

In [13]:
search_result = search.run(query)
type(search_result)

str

In [14]:
search_result = eval(search_result)
type(search_result)

list

In [15]:
search_result

['데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 \u200d♂️ ...more 데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 ...',
 '테디노트 X 패스트캠퍼스 "RAG 비법노트" · 환경 설정 (Mac) · 환경 설정 (Windows). LocalModels. GGUF · HuggingFace gguf 파일을 Ollama 로딩 · TeddyNote.',
 '이 글에서는 LangChain 의 Agent 프레임워크를 활용하여 복잡한 검색과 데이터 처리 작업을 수행하는 방법을 소개합니다. LangSmith 를 사용하여 Agent의 추론 단계를 추적 ...',
 'Teddy Lee (이경록) ... 기업용 AI 솔루션 · AI 교육 · 에이전트 빌더 플랫폼을 만듭니다. LangChain Global Ambassador · 테디노트 운영 · RAG / Agent 시스템 빌더.',
 '테디노트 ; 한줄 소개: 블로그와 유튜브 "테디노트"를 운영하고 있으며, "파이썬 딥러닝 텐서플로"를 집필하였습니다. 데이터분석과 AI를 사랑하고 지식공유에 활발히 참여 ...',
 '경력 ; Ambassador. LangChain. 2025년 2월 – 현재 1년 6개월. 대한민국 서울 ; Content Creator. 테디노트 TeddyNote. 2020년 5월 – 현재 6년 3개월 ; CEO / Founder.',
 '환경 설정 (Windows) · git 설치 · PowerShell Policy 적용 · pyenv 설치 · python 설치 · Poetry 설치 · 실습코드 다운로드 · Visual Studio Code 설치.',
 '데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 \u200d♂️ "테디노트의 RAG 비법노트" 랭체인 강의: ...',
 '데이터 분석, 머신러닝, 딥러닝, LLM 에 대한 내용을 다룹니다. 연구보다는 개발에 관심이 많습니다 \u

# 03. 구조화된 답변을 다음 체인의 입력으로 추가하기

In [16]:
search_result_string = '\n'.join(search_result)

In [17]:
answer

EmailSummary(person='테디', company='테디노트', email='teddy@teddynote.com', subject='RAG 솔루션 시연 관련 미팅 제안', summary='테디노트의 테디가 이은채 대리님에게 AI 및 RAG 솔루션 시연을 위한 미팅을 제안하며, 솔루션의 기능과 적용 방안을 설명하고 협력 가능성을 논의하고자 한다.', date='7월 18일 오전 10시')

In [18]:
from langchain_core.prompts import PromptTemplate

report_prompt = PromptTemplate.from_template(
    '''당신은 이메일의 주요 정보를 바탕으로 요약 정리해 주는 전문가입니다.
    당신의 임무는 다음의 이메일 정보를 바탕으로 보고서 형식의 요약을 작성하는 것입니다.
    주어진 정보를 기반으로 양식(format)에 맞추어 요약을 작성해 주세요.
    
    #Information:
    - Sender: {sender}
    - Additional Information about sender: {additional_information}
    - Company: {company}
    - Email: {email}
    - Subject: {subject}
    - Summary: {summary}
    - Date: {date}
    
    #Format(in markdown format):
    * 보낸 사람:
    - (보낸 사람의 이름, 회사 정보)

    * 이메일 주소:
    - (보낸 사람의 이메일 주소)

    * 보낸 사람과 관련하여 검색된 추가 정보:
    - (검색된 추가 정보)

    * 주요 내용:
    - (이메일 제목, 요약)

    * 일정:
    - (미팅 날짜 및 시간)

    #Answer:
    '''
)

In [19]:
from langchain_core.output_parsers import StrOutputParser

report_chain = (
    report_prompt | ChatOpenAI(model='gpt-4-turbo', temperature=0) | StrOutputParser()
)

In [20]:
report_response = report_chain.invoke(
    {
        'sender': answer.person,
        'additional_information': search_result_string,
        'company': answer.company,
        'email': answer.email,
        'subject': answer.subject,
        'summary': answer.summary,
        'date': answer.date,
    }
)

In [21]:
print(report_response)

* 보낸 사람:
  - 테디, 테디노트

* 이메일 주소:
  - teddy@teddynote.com

* 보낸 사람과 관련하여 검색된 추가 정보:
  - 테디는 데이터 분석, 머신러닝, 딥러닝, LLM에 대한 전문 지식을 가지고 있으며, 연구보다는 개발에 더 큰 관심을 가지고 있습니다. 테디노트의 운영자이며, "파이썬 딥러닝 텐서플로"의 저자입니다. 또한 LangChain의 Global Ambassador로 활동 중이며, RAG 및 Agent 시스템 빌더로도 활동하고 있습니다.

* 주요 내용:
  - 제목: RAG 솔루션 시연 관련 미팅 제안
  - 요약: 테디노트의 테디가 이은채 대리님에게 AI 및 RAG 솔루션 시연을 위한 미팅을 제안하며, 솔루션의 기능과 적용 방안을 설명하고 협력 가능성을 논의하고자 합니다.

* 일정:
  - 7월 18일 오전 10시
